# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their @id
print("Available Record Sets in this dataset:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")

# For demonstration, retrieve fields (columns) for each record set
record_set_fields = {}
for rs in record_sets:
    print(f"\nRecord Set '{rs.name}' (@id={rs.id}) fields (columns):")
    columns = rs.fields
    for field in columns:
        print(f"  - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    record_set_fields[rs.id] = columns

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs.id for rs in dataset.record_sets]
print(f"Record sets available for extraction: {record_set_ids}\n")

# Extract data from each record set into pandas DataFrames
dataframes = dict()
for record_set_id in record_set_ids:
    # Load all records for this record set (usually this is a table)
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows for record set {record_set_id}")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# For demonstration, display the columns of the first record set (if available):
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"\nColumns in record set {example_record_set_id}:\n{dataframes[example_record_set_id].columns.tolist()}")

    # Show the first few rows
    dataframes[example_record_set_id].head()
else:
    print("No record sets available in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example: Let's use the first record set for exploration
record_set_id = record_set_ids[0] if record_set_ids else None

if record_set_id:
    df = dataframes[record_set_id]
    # Try to detect a potential numeric field (e.g. 'MW', 'molecular_weight', etc.)
    from collections.abc import Iterable
    
    # Guess the numeric field by looking for columns of type float, int or containing keywords
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_candidates:
        # Try to guess by keyword
        for w in ['MW', 'molecular_weight', 'value', 'abundance', 'peptides', 'count', 'coverage', 'score']:
            for col in df.columns:
                if w.lower() in col.lower():
                    numeric_candidates.append(col)
    
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}\n")
        try:
            # Drop NA
            df = df.dropna(subset=[numeric_field])

            # Set a threshold at 90th percentile
            threshold = df[numeric_field].quantile(0.9)
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold:.2f} (90th percentile)")

            # Normalize
            filtered_df[f"{numeric_field}_normalized"] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            )
            print(f"\nNormalized {numeric_field} (z-score):")
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Try to group by a category
            possible_groups = ['description', 'name', 'accession', 'group', 'gene']
            group_field = None
            for fn in possible_groups:
                if fn in df.columns:
                    group_field = fn
                    break

            if group_field:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
                print(f"\nGrouped mean of {numeric_field} by {group_field}:")
                print(grouped_df.head())
        except Exception as e:
            print(f"Error during EDA: {e}")
    else:
        print("No obvious numeric fields detected for EDA.")
else:
    print("No available record set for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Simple visualization of the numeric field (if available)
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_candidates:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[record_set_id][numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} in {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=dataframes[record_set_id][numeric_field].dropna())
    plt.title(f"Boxplot of {numeric_field}")
    plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was loaded using its Croissant schema URL and explored using `mlcroissant`.
- Record sets and fields were identified by their unique `@id` as per the Croissant specification.
- Data extraction was demonstrated into DataFrames, with a sample field used for filtering and normalization.
- Basic visualizations give insight into the distribution of a numeric measurement field.
- Further analysis can be performed leveraging the rich metadata, field types, and additional record sets provided in this dataset.